# Trace Tree Illustrations for PTO

This notebook generates illustrations showing the relationship between:
- **Traces** (genotypes with structured naming)
- **Trace trees** (hierarchical visualization)
- **Phenotypes** (solutions)
- **Mutation** and **Crossover** operators

For multiple problem domains: OneMax, TSP, GP, GE, NN.

In [ ]:
# Setup
import random
import numpy as np
from IPython.display import display, HTML

from pto import run, rnd
from pto.gui.trace_tree import (
    trace_tree,
    trace_tree_diff,
    trace_tree_crossover,
    compare_phenotypes,
    save_dot,
)

# Helper to create search operators for a problem
def make_op(generator, fitness, gen_args=(), fit_args=(), better=min, crossover="crossover_one_point_ind"):
    """Create Op with one-point crossover by default (clearer for illustrations)."""
    op = run(
        generator, fitness,
        gen_args=gen_args, fit_args=fit_args,
        better=better, Solver="search_operators",
    )
    # Switch to one-point crossover for clearer illustrations
    op.crossover_ind = getattr(op, crossover)
    return op

# Set seed for reproducibility (optional)
# random.seed(42)

---
## 1. OneMax (Binary String)

The simplest problem: maximize the number of 1s in a binary string.

**Generator:** `[rnd.choice([0, 1]) for i in range(size)]`

**Trace structure:** Flat — one `choice` per bit position.

In [ ]:
from pto.problems.onemax import generator as onemax_gen, fitness as onemax_fit

SIZE = 8
op_onemax = make_op(onemax_gen, onemax_fit, gen_args=(SIZE,), better=max)

### 1.1 Original Solution

In [ ]:
ind_onemax = op_onemax.create_ind()
print(f"Phenotype: {ind_onemax.pheno}")
print(f"Fitness: {onemax_fit(ind_onemax.pheno)}")
print(f"Trace size: {len(ind_onemax.geno)}")
trace_tree(ind_onemax.geno, view=False)

### 1.2 Mutation Effect

In [ ]:
mutant_onemax = op_onemax.mutate_ind(ind_onemax)
compare_phenotypes(ind_onemax, mutant_onemax, "Original", "Mutant")
print(f"Fitness: {onemax_fit(ind_onemax.pheno)} -> {onemax_fit(mutant_onemax.pheno)}")

In [ ]:
# Diff visualization - red nodes show changed values
trace_tree_diff(ind_onemax.geno, mutant_onemax.geno, view=False, title="OneMax Mutation")

### 1.3 One-Point Crossover

One-point crossover selects a random cut point and takes trace entries from parent 1 before the cut, and from parent 2 after.

In [ ]:
parent1_onemax = op_onemax.create_ind()
parent2_onemax = op_onemax.create_ind()
child_onemax = op_onemax.crossover_ind(parent1_onemax, parent2_onemax)

print(f"Parent 1: {parent1_onemax.pheno}")
print(f"Parent 2: {parent2_onemax.pheno}")
print(f"Child:    {child_onemax.pheno}")

In [ ]:
# One-point crossover visualization - blue=parent1 (before cut), orange=parent2 (after cut)
trace_tree_crossover(
    child_onemax.geno, parent1_onemax.geno, parent2_onemax.geno,
    view=False, title="OneMax One-Point Crossover"
)

---
## 2. TSP (Permutation)

Traveling Salesman Problem: find the shortest tour visiting all cities.

**Generator (Knuth shuffle):** Sequence of swaps via `rnd.choice(range(i, N))`

**Trace structure:** Loop with shrinking choice ranges — repair maintains permutation validity.

In [ ]:
from pto.problems.tsp import generator_knuth as tsp_gen, fitness as tsp_fit, make_problem_data

N_CITIES = 6
dist_matrix = make_problem_data(N_CITIES, random_state=42)
op_tsp = make_op(tsp_gen, tsp_fit, gen_args=(N_CITIES,), fit_args=(dist_matrix,), better=min)

### 2.1 Original Solution

In [ ]:
ind_tsp = op_tsp.create_ind()
print(f"Tour: {ind_tsp.pheno}")
print(f"Distance: {tsp_fit(ind_tsp.pheno, dist_matrix):.4f}")
trace_tree(ind_tsp.geno, view=False)

### 2.2 Mutation with Repair

Mutation changes a choice in the shuffle sequence. Repair ensures the result is still a valid permutation.

In [ ]:
mutant_tsp = op_tsp.mutate_ind(ind_tsp)
compare_phenotypes(ind_tsp, mutant_tsp, "Original", "Mutant")
print(f"Distance: {tsp_fit(ind_tsp.pheno, dist_matrix):.4f} -> {tsp_fit(mutant_tsp.pheno, dist_matrix):.4f}")

In [ ]:
trace_tree_diff(ind_tsp.geno, mutant_tsp.geno, view=False, title="TSP Mutation")

### 2.3 One-Point Crossover

In [ ]:
parent1_tsp = op_tsp.create_ind()
parent2_tsp = op_tsp.create_ind()
child_tsp = op_tsp.crossover_ind(parent1_tsp, parent2_tsp)

print(f"Parent 1: {parent1_tsp.pheno}")
print(f"Parent 2: {parent2_tsp.pheno}")
print(f"Child:    {child_tsp.pheno}")

In [ ]:
trace_tree_crossover(
    child_tsp.geno, parent1_tsp.geno, parent2_tsp.geno,
    view=False, title="TSP One-Point Crossover"
)

---
## 3. Symbolic Regression / GP (Tree)

Genetic Programming: evolve boolean expressions.

**Generator:** Recursive `rnd_expr(depth)` with `rnd.random()` for growth and `rnd.choice()` for terminals/functions.

**Trace structure:** Deep recursive tree matching the expression structure.

In [ ]:
from pto.problems.symbolic_regression import generator as gp_gen, fitness as gp_fit

N_VARS = 3
FUNC_SET = [("and", 2), ("or", 2), ("not", 1)]
TERM_SET = [f"x[{i}]" for i in range(N_VARS)]
MAX_DEPTH = 3

# Dummy fitness args
X_dummy = [[True] * N_VARS]
y_dummy = [True]

op_gp = make_op(gp_gen, gp_fit, gen_args=(FUNC_SET, TERM_SET, MAX_DEPTH), fit_args=(X_dummy, y_dummy), better=min)

### 3.1 Original Expression

In [ ]:
# Generate a few and pick one with interesting structure
ind_gp = max((op_gp.create_ind() for _ in range(10)), key=lambda s: len(s.geno))
print(f"Expression: {ind_gp.pheno}")
print(f"Trace size: {len(ind_gp.geno)}")
trace_tree(ind_gp.geno, view=False)

### 3.2 Mutation (Subtree Change)

In [ ]:
mutant_gp = op_gp.mutate_ind(ind_gp)
compare_phenotypes(ind_gp, mutant_gp, "Original", "Mutant")

In [ ]:
trace_tree_diff(ind_gp.geno, mutant_gp.geno, view=False, title="GP Mutation")

### 3.3 One-Point Crossover

In [ ]:
parent1_gp = max((op_gp.create_ind() for _ in range(5)), key=lambda s: len(s.geno))
parent2_gp = max((op_gp.create_ind() for _ in range(5)), key=lambda s: len(s.geno))
child_gp = op_gp.crossover_ind(parent1_gp, parent2_gp)

print(f"Parent 1: {parent1_gp.pheno}")
print(f"Parent 2: {parent2_gp.pheno}")
print(f"Child:    {child_gp.pheno}")

In [ ]:
trace_tree_crossover(
    child_gp.geno, parent1_gp.geno, parent2_gp.geno,
    view=False, title="GP One-Point Crossover"
)

---
## 4. Grammatical Evolution (Grammar-Guided)

GE: evolve expressions by expanding grammar rules.

**Generator:** Recursive expansion with `rnd.choice()` for rule selection.

**Trace structure:** Matches grammar derivation tree.

In [ ]:
from pto.problems.grammatical_evolution import generator as ge_gen, fitness as ge_fit, grammar, make_training_data

N_VARS_GE = 3
grammar["<varidx>"] = [[str(i)] for i in range(N_VARS_GE)]
X_train_ge, y_train_ge = make_training_data(20, N_VARS_GE)

op_ge = make_op(ge_gen, ge_fit, gen_args=(grammar,), fit_args=(X_train_ge, y_train_ge), better=min)

### 4.1 Original Expression

In [ ]:
ind_ge = max((op_ge.create_ind() for _ in range(10)), key=lambda s: len(s.geno))
print(f"Expression: {ind_ge.pheno}")
print(f"Trace size: {len(ind_ge.geno)}")
trace_tree(ind_ge.geno, view=False)

### 4.2 Mutation

In [ ]:
mutant_ge = op_ge.mutate_ind(ind_ge)
compare_phenotypes(ind_ge, mutant_ge, "Original", "Mutant")

In [ ]:
trace_tree_diff(ind_ge.geno, mutant_ge.geno, view=False, title="GE Mutation")

### 4.3 One-Point Crossover

In [ ]:
parent1_ge = max((op_ge.create_ind() for _ in range(5)), key=lambda s: len(s.geno))
parent2_ge = max((op_ge.create_ind() for _ in range(5)), key=lambda s: len(s.geno))
child_ge = op_ge.crossover_ind(parent1_ge, parent2_ge)

print(f"Parent 1: {parent1_ge.pheno}")
print(f"Parent 2: {parent2_ge.pheno}")
print(f"Child:    {child_ge.pheno}")

In [ ]:
trace_tree_crossover(
    child_ge.geno, parent1_ge.geno, parent2_ge.geno,
    view=False, title="GE One-Point Crossover"
)

---
## 5. Neural Network (Weight Vector)

NN: evolve neural network weights.

**Generator:** Nested loops with `rnd.randint()` for hidden size and `rnd.normalvariate()` for weights.

**Trace structure:** Two-level — hidden layer weights and output layer weights.

In [ ]:
from pto.problems.neural_network import generator as nn_gen, fitness as nn_fit, make_training_data as nn_make_data

N_INPUTS = 3
MAX_HIDDEN = 4
N_OUTPUTS = N_INPUTS
X_train_nn, y_train_nn = nn_make_data(10, N_INPUTS)

op_nn = make_op(
    nn_gen, nn_fit,
    gen_args=(N_INPUTS, MAX_HIDDEN, N_OUTPUTS),
    fit_args=(X_train_nn, y_train_nn),
    better=min
)

### 5.1 Original Network

In [ ]:
ind_nn = op_nn.create_ind()
n_hidden = len(ind_nn.pheno[0])
print(f"Network structure: {N_INPUTS} -> {n_hidden} -> {N_OUTPUTS}")
print(f"Trace size: {len(ind_nn.geno)}")
trace_tree(ind_nn.geno, view=False)

### 5.2 Mutation (Weight Perturbation)

In [ ]:
mutant_nn = op_nn.mutate_ind(ind_nn)
print(f"Original hidden size: {len(ind_nn.pheno[0])}")
print(f"Mutant hidden size:   {len(mutant_nn.pheno[0])}")

In [ ]:
trace_tree_diff(ind_nn.geno, mutant_nn.geno, view=False, title="NN Mutation")

### 5.3 One-Point Crossover

In [ ]:
parent1_nn = op_nn.create_ind()
parent2_nn = op_nn.create_ind()
child_nn = op_nn.crossover_ind(parent1_nn, parent2_nn)

print(f"Parent 1 hidden: {len(parent1_nn.pheno[0])}")
print(f"Parent 2 hidden: {len(parent2_nn.pheno[0])}")
print(f"Child hidden:    {len(child_nn.pheno[0])}")

In [ ]:
trace_tree_crossover(
    child_nn.geno, parent1_nn.geno, parent2_nn.geno,
    view=False, title="NN One-Point Crossover"
)

---
## 6. Export Figures

Save selected figures for the paper.

In [ ]:
# Save as .dot files (can be rendered later with: dot -Tpdf file.dot -o file.pdf)
# save_dot(trace_tree(ind_onemax.geno, view=False), "fig_onemax_tree")
# save_dot(trace_tree_diff(ind_onemax.geno, mutant_onemax.geno, view=False), "fig_onemax_mutation")
# save_dot(trace_tree_crossover(child_gp.geno, parent1_gp.geno, parent2_gp.geno, view=False), "fig_gp_crossover")

# Or render directly to PDF/PNG (requires Graphviz binaries on PATH):
# trace_tree_diff(ind_onemax.geno, mutant_onemax.geno, filename="fig_onemax_mutation", view=False)

---
## 7. Custom Exploration

Use this section to generate and refine specific examples.

In [ ]:
# Your custom code here
# Example: find a mutation that flips multiple bits
# for _ in range(20):
#     m = op_onemax.mutate_ind(ind_onemax)
#     diff = sum(a != b for a, b in zip(ind_onemax.pheno, m.pheno))
#     if diff >= 3:
#         print(f"Found mutation with {diff} changes")
#         trace_tree_diff(ind_onemax.geno, m.geno, view=False)
#         break